In [25]:
# =====================================================
# 1. IMPORT THƯ VIỆN
# =====================================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt
import seaborn as sns

# =====================================================
# 2. ĐỌC DỮ LIỆU
# =====================================================

df = pd.read_csv('../data/raw/online_retail_II.csv')

# Xem dữ liệu
print(df.head())

# =====================================================
# 3. KIỂM TRA DỮ LIỆU
# =====================================================

print(df.info())

# Kiểm tra missing values
print(df.isnull().sum())

# Kiểm tra duplicate
print(df.duplicated().sum())

# Thống kê mô tả
print(df.describe())

# =====================================================
# 4. LÀM SẠCH DỮ LIỆU
# =====================================================

# -------------------------
# 4.1 Xử lý missing values
# -------------------------

df = df.dropna()

# -------------------------
# 4.2 Xử lý duplicate
# -------------------------

df = df.drop_duplicates()

# -------------------------
# 4.3 Xử lý dữ liệu bất thường
# -------------------------

# Loại bỏ quantity <= 0
df = df[df['Quantity'] > 0]

# Loại bỏ price <= 0
df = df[df['Price'] > 0]
# =====================================================
# CLEAN STOCKCODE
# =====================================================

# Chuyển về string
df['StockCode'] = df['StockCode'].astype(str)

# Xóa khoảng trắng đầu/cuối
df['StockCode'] = df['StockCode'].str.strip()

# Chuyển về uppercase
df['StockCode'] = df['StockCode'].str.upper()

# =====================================================
# DANH SÁCH MÃ KHÔNG HỢP LỆ
# =====================================================

invalid_codes = [
    'POST',
    'DOT',
    'M',
    'BANK CHARGES',
    'AMAZONFEE',
    'CRUK',
    'D',
    'C2',
    'PADS',
    'ADJUST',
    'ADJUST2',
    'TEST001',
    'TEST002',
    'SP1002'
]

# =====================================================
# LOẠI BỎ MÃ KHÔNG HỢP LỆ
# =====================================================

df = df[
    ~df['StockCode'].isin(invalid_codes)
]

# =====================================================
# LOẠI BỎ CÁC MÃ TEST
# =====================================================

df = df[
    ~df['StockCode'].str.startswith('TEST')
]

# =====================================================
# KIỂM TRA
# =====================================================

print(
    df[df['StockCode'].isin(invalid_codes)]
)

print('Done cleaning stock codes!')

# =====================================================
# 5. CHUYỂN ĐỔI DỮ LIỆU
# =====================================================

# Chuyển InvoiceDate sang datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# =====================================================
# 6. TẠO CÁC ĐẶC TRƯNG HÀNH VI BÁN
# =====================================================

# -----------------------------------------------------
# 6.1 Tổng số lượng bán
# -----------------------------------------------------

total_sales = df.groupby('StockCode')['Quantity'].sum()

# -----------------------------------------------------
# 6.2 Tần suất bán
# -----------------------------------------------------

sale_frequency = df.groupby('StockCode').size()

# -----------------------------------------------------
# 6.3 Doanh thu sản phẩm
# -----------------------------------------------------

df['Revenue'] = df['Quantity'] * df['Price']

revenue = df.groupby('StockCode')['Revenue'].sum()

# -----------------------------------------------------
# 6.4 Độ biến động doanh số
# -----------------------------------------------------

sales_variance = df.groupby('StockCode')['Quantity'].var()

# -----------------------------------------------------
# 6.5 Số ngày không bán
# -----------------------------------------------------

latest_date = df['InvoiceDate'].max()

last_sale_date = df.groupby('StockCode')['InvoiceDate'].max()

days_since_last_sale = (
    latest_date - last_sale_date
).dt.days

# -----------------------------------------------------
# 6.6 Tạo tồn kho giả lập
# -----------------------------------------------------

avg_inventory = total_sales.apply(
    lambda x: np.random.randint(
        max(10, int(x * 0.5)),
        max(20, int(x * 2))
    )
)

# -----------------------------------------------------
# 6.7 Tốc độ quay vòng tồn kho
# -----------------------------------------------------

stock_turnover = total_sales / avg_inventory

# =====================================================
# 7. GỘP FEATURE THÀNH DATAFRAME
# =====================================================

features_df = pd.DataFrame({
    'total_sales': total_sales,
    'sale_frequency': sale_frequency,
    'revenue': revenue,
    'sales_variance': sales_variance,
    'days_since_last_sale': days_since_last_sale,
    'avg_inventory': avg_inventory,
    'stock_turnover': stock_turnover
})

# Xử lý NaN
features_df = features_df.fillna(0)

# =====================================================
# 8. RESET INDEX
# =====================================================

features_df.reset_index(inplace=True)

# Đổi tên cột
features_df.rename(
    columns={'StockCode': 'Product_ID'},
    inplace=True
)

# =====================================================
# 9. CHUẨN HÓA DỮ LIỆU
# =====================================================

scaler = StandardScaler()

scaled_columns = [
    'total_sales',
    'sale_frequency',
    'revenue',
    'sales_variance',
    'days_since_last_sale',
    'avg_inventory',
    'stock_turnover'
]

features_df[scaled_columns] = scaler.fit_transform(
    features_df[scaled_columns]
)

# =====================================================
# 10. KIỂM TRA KẾT QUẢ
# =====================================================

print(features_df.head())

print(features_df.shape)

# =====================================================
# 11. LƯU DỮ LIỆU ĐÃ XỬ LÝ
# =====================================================

features_df.to_csv(
    '../data/processed/processed_retail_data.csv',
    index=False
)

print('Đã lưu dữ liệu thành công!')

  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434    79323P                   PINK CHERRY LIGHTS        12   
2  489434    79323W                  WHITE CHERRY LIGHTS        12   
3  489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   

           InvoiceDate  Price  Customer ID         Country  
0  2009-12-01 07:45:00   6.95      13085.0  United Kingdom  
1  2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
2  2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
3  2009-12-01 07:45:00   2.10      13085.0  United Kingdom  
4  2009-12-01 07:45:00   1.25      13085.0  United Kingdom  
<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      106737